# Session 14: Machine learning: the workflow

Prepare the Nile Microfinance loans with pandas, then fit a first model and compare it with the baseline. The pandas cells run exactly as shown. Scores from scikit-learn will be close to the lesson, not identical.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Load the loans
Read `loans.csv` into a DataFrame called `loans` and print its shape and the default rate.

In [ ]:
import pandas as pd

loans = pd.read_csv('loans.csv')
print(loans.shape)
print(loans.default.mean())

In [ ]:
assert loans.shape == (8000, 18), 'Check the file name'
assert abs(loans.default.mean() - 0.1255) < 0.001, 'Use the mean of the default column'
print('Task 1 correct')

## Task 2. Fix the impossible ages
An age under 18 or over 100 is an error. Replace each with the median of the good ages. Store the mask in `bad_age`.

In [ ]:
bad_age = (loans.age < 18) | (loans.age > 100)
print(bad_age.sum())
loans.loc[bad_age, 'age'] = int(loans.loc[~bad_age, 'age'].median())
print(loans.age.min(), loans.age.max())

In [ ]:
assert bad_age.sum() == 2, 'Two ages should be flagged'
assert loans.age.between(18, 100).all(), 'Every age should now be between 18 and 100'
print('Task 2 correct')

## Task 3. Missing incomes
Add a column `income_missing` that is 1 where income is blank and 0 elsewhere. Then fill the blanks in `monthly_income_ugx` with the median income, and store that median in `median_income`.

In [ ]:
loans['income_missing'] = loans.monthly_income_ugx.isna().astype(int)
median_income = loans.monthly_income_ugx.median()
loans['monthly_income_ugx'] = loans.monthly_income_ugx.fillna(median_income)
print(median_income, loans.income_missing.sum())

In [ ]:
assert loans.income_missing.sum() == 379, 'Make the flag before you fill'
assert loans.monthly_income_ugx.isna().sum() == 0, 'Fill every blank income'
assert median_income == 642600.0, 'Use the median of monthly_income_ugx'
print('Task 3 correct')

## Task 4. The burden feature
Add `burden` = loan amount times 1.15, divided by term in months, divided by monthly income.

In [ ]:
loans['burden'] = loans.loan_amount_ugx * 1.15 / loans.term_months / loans.monthly_income_ugx
print(loans.burden.median().round(2))

In [ ]:
assert round(loans.burden.median(), 2) == 0.29, 'Check the order of the divisions'
print('Task 4 correct')

## Task 5. Features and target
Build `X` from these twelve columns of `loans`: `age`, `monthly_income_ugx`, `income_missing`, `loan_amount_ugx`, `term_months`, `previous_loans`, `previous_late_payments`, `group_loan`, `savings_balance_ugx`, `has_mobile_money`, `distance_to_branch_km` and `burden`. Build `y` from `default`. Leave `collection_calls` out. Session 16 explains why.

In [ ]:
features = ['age', 'monthly_income_ugx', 'income_missing',
            'loan_amount_ugx', 'term_months', 'previous_loans',
            'previous_late_payments', 'group_loan',
            'savings_balance_ugx', 'has_mobile_money',
            'distance_to_branch_km', 'burden']
X = loans[features]
y = loans['default']
print(X.shape, y.shape)

In [ ]:
assert X.shape == (8000, 12), 'X should have 12 columns and 8000 rows'
assert 'collection_calls' not in X.columns, 'Leave collection_calls out'
assert X.isna().sum().sum() == 0, 'No blanks should remain. Did you fill the incomes in Task 3?'
assert len(y) == 8000, 'y should be the default column'
print('Task 5 correct')

## Task 6. Split and score the baseline
Split with `test_size=0.25`, `stratify=y` and `random_state=42`. Fit a `DummyClassifier(strategy='most_frequent')` and store its test accuracy in `base_acc`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
base_acc = baseline.score(X_test, y_test)
print(len(X_test), base_acc)

In [ ]:
assert 0.80 <= base_acc <= 0.92, 'The baseline accuracy should be near 0.87. Score it on X_test and y_test'
assert len(X_test) == 2000, 'Use test_size=0.25 so the test part has 2000 loans'
print('Task 6 correct')

## Task 7. A first model
Fit `DecisionTreeClassifier(max_depth=3, random_state=42)` on the training set, predict on the test set and store the test accuracy in `model_acc`. Print it beside `base_acc`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

first = DecisionTreeClassifier(max_depth=3, random_state=42)
first.fit(X_train, y_train)
pred = first.predict(X_test)
model_acc = first.score(X_test, y_test)
print('baseline', round(base_acc, 3), 'model', round(model_acc, 3))

In [ ]:
assert 0.80 <= model_acc <= 0.92, 'The test accuracy should be near 0.87 to 0.89. Score the test part, not the training part'
assert len(pred) == 2000, 'Predict on X_test'
print('Task 7 correct')

## Task 8. Reflect and choose your dataset
Write one sentence below: did the model beat the baseline, and by how much? What does that tell you about accuracy on rare events? Then choose your capstone dataset and write its name: Lakeside Mart, Nile Microfinance loans, District Clinics visits or Lakeside Mart feedback. Check that it can answer the question you wrote in Session 13.